# NaviGator AI from Python

A starter notebook from Harness Kits, Kit 40. It uses NaviGator's **local** models, which are approved for open, sensitive, and restricted data.

Start Jupyter from a terminal where `NAVIGATOR_TOOLKIT_API_KEY` is set (Kit 00), so the key never appears in this notebook.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("NAVIGATOR_BASE_URL", "https://api.ai.it.ufl.edu/v1"),
    api_key=os.environ["NAVIGATOR_TOOLKIT_API_KEY"],   # read from the environment, never typed here
)

# Which models can this key use? Check that the local ones used below are there.
ids = {m.id for m in client.models.list()}
print(len(ids), "models")
{m: m in ids for m in ["meta-muse-glimmer-30b", "gpt-oss-120b", "nomic-embed-text-v1.5", "whisper-large-v3"]}

## 1. Ask a question

In [ ]:
reply = client.chat.completions.create(
    model="meta-muse-glimmer-30b",
    messages=[{"role": "user", "content": "In one sentence, what is a p-value?"}],
    temperature=0,
)
print(reply.choices[0].message.content)
print(reply.usage)   # tokens used; this is what counts against your credit

## 2. Code open-ended survey responses

Each response gets exactly one code from a fixed list, plus a reason. Labels outside the list are marked `ERROR` instead of being trusted. Replace the sample responses with your own column.

In [ ]:
import json
import pandas as pd

df = pd.DataFrame({"response": [
    "The bus only comes twice a day so I miss appointments.",
    "My copay went up again this year and I skipped refills.",
    "The doctor rushed me and didn't answer my questions.",
    "I work nights and the clinic closes at 5.",
    "Honestly the care was great, no complaints.",
    "I couldn't get a ride after my car broke down.",
    "Insurance wouldn't cover the specialist.",
    "They never called me back to reschedule.",
    "I don't trust them since they got my diagnosis wrong.",
    "Parking at the hospital costs twenty dollars.",
]})

CODES = ["transportation", "cost", "scheduling", "trust or communication", "positive", "other"]

def code_response(text):
    reply = client.chat.completions.create(
        model="meta-muse-glimmer-30b",
        temperature=0,
        messages=[
            {"role": "system", "content":
                "You code survey responses about health care access. "
                f"Choose exactly one code from this list: {', '.join(CODES)}. "
                'Reply with JSON only: {"code": "...", "reason": "..."}'},
            {"role": "user", "content": text},
        ],
    )
    raw = reply.choices[0].message.content
    try:
        out = json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
    except ValueError:
        return {"code": "ERROR", "reason": raw[:200]}
    if out.get("code") not in CODES:
        out = {"code": "ERROR", "reason": f"unexpected code: {out.get('code')}"}
    return out

coded = pd.concat([df, df["response"].apply(code_response).apply(pd.Series)], axis=1)
coded.to_csv("coded_responses.csv", index=False)   # save results so you don't pay twice
coded

## 3. Check the codes against your own

Code a sample yourself, blind to the model's answers, then compare. Report the agreement rate, and read every disagreement.

In [ ]:
sample = coded.sample(n=min(10, len(coded)), random_state=1)
sample[["response"]].to_csv("my_codes_to_fill.csv", index=False)
# Fill in a my_code column in my_codes_to_fill.csv, save it, then:
# mine = pd.read_csv("my_codes_to_fill.csv")
# check = sample.merge(mine, on="response")
# print("agreement:", (check["code"] == check["my_code"]).mean())
# check[check["code"] != check["my_code"]]

## 4. Search by meaning with embeddings

Embeddings turn text into vectors, so you can find responses that mean the same thing in different words. Test with a question whose answer you already know.

In [ ]:
import numpy as np

def embed(texts):
    data = client.embeddings.create(model="nomic-embed-text-v1.5", input=texts).data
    v = np.array([d.embedding for d in data])
    return v / np.linalg.norm(v, axis=1, keepdims=True)

vectors = embed(["search_document: " + t for t in coded["response"]])
query = embed(["search_query: problems with transportation to appointments"])[0]
coded.assign(similarity=vectors @ query).sort_values("similarity", ascending=False)[["response", "similarity"]].head(3)

## 5. Transcribe an interview

`whisper-large-v3` runs locally. It doesn't label speakers, so add those yourself. Put an audio file (for example `interview.wav`) next to this notebook.

In [ ]:
with open("interview.wav", "rb") as audio:
    transcript = client.audio.transcriptions.create(model="whisper-large-v3", file=audio)
print(transcript.text)